# Music Library Database

**Overview**

The `database.py` module provides a complete SQLAlchemy-based interface for managing a music library database. It handles:

1. **Storing** - Audio file metadata in a structured relational database
2. **Scanning** - Automatic extraction of metadata from audio files
3. **Querying** - Flexible search with artist, album, BPM, year, genre filters
4. **Updating** - Both full scans and incremental updates
5. **Flexibility** - SQLite by default, MySQL support via configuration

**Database Models**

The database uses three main models (defined in `models.py`):

- `Track` - Individual audio files with metadata (title, artist, album, BPM, duration, etc.)
- `Artist` - Artist information including Discogs data
- `Release` - Album/release information linked to artists and tracks

**Key Features**

- Supports multiple audio formats: MP3, FLAC, M4A, MP4, OGG, WAV
- Extracts audio metadata using mutagen library
- SQLite by default, but supports MySQL via connection URL
- Automatic relationship management between artists, releases, and tracks

**Key Functions:**
- `init_db()` - Create database tables
- `scan_library()` - Full library scan
- `incremental_update()` - Update only new/modified files
- `query()` - Search with multiple filters
- `get_audio_file_metadata()` - Extract metadata from a single file


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
# import sys
from pathlib import Path
from sqlalchemy import inspect as sqla_inspect

# # Add project root to path
# sys.path.insert(0, str(Path.cwd().parent / 'src'))

from aft.db.database import (
    init_db,
    get_database_url,
    get_session_factory,
    get_audio_file_metadata,
    scan_library,
    incremental_update,
    query,
    AUDIO_EXTENSIONS,
    # get_database_url, engine
)
from aft.db.models import Track
# Artist, Release
import logging

# Set up logging to see what's happening
logging.basicConfig(
    level=logging.INFO,
    format='%(name)s - %(levelname)s - %(message)s'
)

## Initialize database (first time only)

In [3]:
# Check the default database URL
print(f"Current working directory: {os.getcwd()}")
print(f"Database URL: {get_database_url()}")

Current working directory: c:\Users\aleja\Projects\audio-files-tagging\notebooks
Database URL: sqlite:///data/library.db


In [4]:
# When running from notebooks/, specify absolute path to database in project root
db_path = str(Path.cwd().parent / 'data' / 'library.db')
print(f"Using database: {db_path}")
init_db(db_path)

aft.db.database - INFO - Database initialized at c:\Users\aleja\Projects\audio-files-tagging\data\library.db


Using database: c:\Users\aleja\Projects\audio-files-tagging\data\library.db


In [5]:
print("Supported audio formats:")
for ext in sorted(AUDIO_EXTENSIONS):
    print(f"  {ext}")

Supported audio formats:
  .flac
  .m4a
  .mp3
  .mp4
  .ogg
  .wav


### Display Track model structure

In [6]:
inspector = sqla_inspect(Track)
print("Track Model Columns:")
print("-" * 50)
for column in inspector.columns:
    nullable = "NULL" if column.nullable else "NOT NULL"
    print(f"{column.name:20} {str(column.type):20} {nullable}")
    
print("\nKey Properties:")
print("- id: Primary key")
print("- file_path: Unique file system path (UNIQUE constraint)")
print("- title, artist, album: Basic metadata")
print("- bpm, key: Musical properties") 
print("- duration: Track length in seconds")
print("- bitrate, sample_rate: Audio quality info")
print("- tags_json: Full JSON of all tags from file")
print("- last_modified: File modification timestamp")
print("- release_id: Foreign key to Release table")

Track Model Columns:
--------------------------------------------------
id                   INTEGER              NOT NULL
title                VARCHAR(255)         NOT NULL
artist               VARCHAR(255)         NULL
album                VARCHAR(255)         NULL
position             VARCHAR(32)          NULL
duration             FLOAT                NULL
file_path            VARCHAR(1024)        NOT NULL
bpm                  FLOAT                NULL
key                  VARCHAR(32)          NULL
genre                VARCHAR(255)         NULL
year                 INTEGER              NULL
bitrate              INTEGER              NULL
sample_rate          INTEGER              NULL
tags_json            JSON                 NULL
last_modified        DATETIME             NULL
discogs_id           INTEGER              NULL
release_id           INTEGER              NULL

Key Properties:
- id: Primary key
- file_path: Unique file system path (UNIQUE constraint)
- title, artist, album: B

### Audio Metadata Extraction

The `get_audio_file_metadata()` function extracts:
- **Basic tags**: title, artist, album, year, genre
- **Musical properties**: BPM, key
- **Technical info**: duration, bitrate, sample_rate
- **File info**: file path, last modified timestamp
- **Raw tags**: Complete JSON dump of all tags

In [7]:
# Example: Extract metadata from a single file
metadata = get_audio_file_metadata(Path(r"D:\Soulseek Downloads\complete\Dragutesku, Search DiP - Prelude [DRGL002]\Dragutesku, Search DiP - Prelude (Original Mix).mp3"))
if metadata:
    print(f"Title: {metadata['title']}")
    print(f"Artist: {metadata['artist']}")
    print(f"BPM: {metadata['bpm']}")
    print(f"Duration: {metadata['duration']:.2f} seconds")
    print(f"Bitrate: {metadata['bitrate']} bps")

print("Metadata extraction example (commented out)")

Title: Prelude (Original Mix)
Artist: Dragutesku, Search DiP
BPM: 126.0
Duration: 457.14 seconds
Bitrate: 320000 bps
Metadata extraction example (commented out)


## Library Scan

`scan_library()` performs a complete scan of a directory tree. It:
- Finds all audio files (mp3, flac, m4a, mp4, ogg, wav)
- Extracts metadata using mutagen (title, artist, album, BPM, duration, bitrate, etc.)
- Adds new tracks or updates existing ones
- Commits every 100 files for efficiency

In [8]:
# Example: scan a music directory
# Replace with your actual music folder path
db_path = str(Path.cwd().parent / 'data' / 'library.db')
base_path = r"C:\Users\aleja\Projects\audio-files-tagging\data" # "D:\Music Collection"
scan_library(
    base_path=base_path,
    db_path=db_path
)

aft.db.database - INFO - Starting full library scan of C:\Users\aleja\Projects\audio-files-tagging\data
aft.db.database - INFO - Found 2 audio files
aft.db.database - INFO - Scan complete. Added: 2, Updated: 0


## Incremental Update

`incremental_update()` is faster for subsequent scans. It only processes:
- New files that weren't in the database
- Files that have been modified since the last scan (checks timestamp)

In [ ]:
# Example: perform incremental update
db_path = str(Path.cwd().parent / 'data' / 'library.db')
base_path = r"C:\Users\aleja\Projects\audio-files-tagging\data" # "D:\Music Collection"

updated_files = incremental_update(base_path=base_path, db_path=db_path)
print(f"Updated {len(updated_files)} files")

aft.db.database - INFO - Starting incremental update of C:\Users\aleja\Projects\audio-files-tagging\data
aft.db.database - INFO - Incremental update complete. Added: 0, Updated: 1


Updated 1 files


## Querying the Database

The `query()` function provides powerful search capabilities with multiple filters. Let's look at various query patterns:

In [48]:
SessionFactory = get_session_factory(db_path)
session = SessionFactory()

# Example 1: Find all tracks by a specific artist (partial match, case-insensitive)
tracks = query(artist="NTFO", session=session)
for track in tracks[:5]:  # Show first 5
    print(f"{track.title} - {track.artist} ({track.album})")

# Example 2: Find tracks in a specific BPM range (great for DJs)
tracks = query(bpm_range=(120, 130), session=session)
print(f"Found {len(tracks)} tracks between 120-130 BPM")

# Example 3: Complex query combining multiple filters
tracks = query(
    artist="Daft Punk",
    bpm_range=(110, 130),
    year_range=(2000, 2020),
    limit=50,
    session=session
)

# Example 4: Text search across artist, album, and title
tracks = query(text_search="electronic", session=session)

# Example 5: Filter by genre
tracks = query(genre="Rock", limit=10, session=session)


aft.db.database - INFO - Query returned 1 results
aft.db.database - INFO - Query returned 1 results
aft.db.database - INFO - Query returned 0 results
aft.db.database - INFO - Query returned 2 results
aft.db.database - INFO - Query returned 0 results


Acelstalalt - NTFO (www.electronicfresh.com)
Found 1 tracks between 120-130 BPM


### Working with Sessions

For more advanced database operations, you can work directly with SQLAlchemy sessions:

In [49]:
# Create a session factory
db_path = str(Path.cwd().parent / 'data' / 'library.db')
SessionFactory = get_session_factory(db_path)
session = SessionFactory()

try:
    # Example: Get total number of tracks
    track_count = session.query(Track).count()
    print(f"Total tracks in database: {track_count}")
    
    # Example: Get all unique artists (from Track table)
    unique_artists = session.query(Track.artist).distinct().all()
    print(f"Unique artists: {len(unique_artists)}")
    
    # Example: Get tracks ordered by BPM
    fast_tracks = session.query(Track).filter(Track.bpm.isnot(None)).order_by(Track.bpm.desc()).limit(10).all()
    for track in fast_tracks:
        print(f"{track.title} - {track.bpm} BPM")
    
finally:
    session.close()

Total tracks in database: 2
Unique artists: 2
Acelstalalt - 126.0 BPM


## Delete Records

In [7]:
from aft.db.models import Track
from aft.db.database import get_session_factory
from pathlib import Path

db_path = str(Path.cwd().parent / 'data' / 'library.db')
SessionFactory = get_session_factory(db_path)
session = SessionFactory()

try:
    # # Method 1: Delete a specific track by ID
    # track = session.query(Track).filter(Track.id == 123).first()
    # if track:
    #     session.delete(track)
    #     session.commit()
    #     print(f"Deleted track: {track.title}")
    
    # # Method 2: Delete tracks matching a condition
    # deleted_count = session.query(Track).filter(
    #     Track.artist == "Artist Name"
    # ).delete()
    # session.commit()
    # print(f"Deleted {deleted_count} tracks")
    
    # # Method 3: Delete tracks by file path
    # session.query(Track).filter(
    #     Track.file_path == r"C:\path\to\file.mp3"
    # ).delete()
    # session.commit()
    
    # # Method 4: Delete tracks in a BPM range
    # session.query(Track).filter(
    #     Track.bpm.between(120, 130)
    # ).delete()
    # session.commit()
    
    # Method 5: Delete all tracks (use with caution!)
    session.query(Track).delete()
    session.commit()
    
finally:
    session.close()